# Pipeline de amostragem para dados desbalanceados — Bloco 2: técnicas e triagem

Este bloco recebe o `contexto` do bloco 1, aplica as técnicas de amostragem **só no treino**, treina o *proxy* (um LightGBM com hiperparâmetros fixos) para cada configuração e compara as técnicas no **teste**. A saída é a **lista curta**: as técnicas que passam para a decisão final no bloco 3.

**Pré-requisitos, na mesma sessão:**

1. As células do bloco 1 executadas e o `contexto` criado por `preparar_bases(...)`.
2. A biblioteca `lightgbm` instalada (é a mesma que a sua `FeatureReduction` usa). O SynapseML só é necessário se alguma configuração passar do limite de memória do *driver*.

| Célula | Conteúdo |
|---|---|
| 1 | Parâmetros do bloco 2 e catálogo das técnicas |
| 2 | Métricas: KS, AUC-ROC, PR-AUC, grade operacional, por safra e bootstrap pareado |
| 3 | *Proxy*: LightGBM em pandas, com o SynapseML como alternativa |
| 4 | Vizinhança: padronização e busca dos vizinhos mais próximos (no *driver* e distribuída) |
| 5 | Análises compartilhadas: Tomek, ENN, NCR, NearMiss, agrupamento e *cross-fitting* |
| 6 | Bases da triagem: treino e teste com identificador de linha, amostra de avaliação do treino |
| 7 | As 16 técnicas |
| 8 | Triagem: grade, execução com retomada, resumo, bootstrap e lista curta |
| 9 | Exemplo de uso |

Convenções do bloco 1 continuam valendo: **positivos** são alvo = 1, **negativos** são alvo = 0; só funções, sem classes; o notebook funciona mesmo com `from pyspark.sql.functions import *` ativo (as funções do Python `min`, `max`, `sum`, `round`, `abs` e `pow` são chamadas como `builtins.min` etc., e a `reduce` como `functools.reduce`).

## Célula 1 — Parâmetros do bloco 2 e catálogo das técnicas

Os parâmetros do bloco 1 (níveis da grade, número de sementes, tamanho da lista curta, taxas de alerta, bootstrap) continuam valendo e podem ser sobrescritos aqui também.

| Parâmetro | Padrão | Função |
|---|---|---|
| `max_features_distancia` | 30 | Máximo de features usadas no cálculo de distâncias; ficam as mais importantes do baseline |
| `k_vizinhos` | 5 | Vizinhos positivos usados para criar pontos sintéticos (SMOTE e variantes) |
| `m_vizinhos_borderline` | 10 | Vizinhos na base toda usados para achar a fronteira (Borderline-SMOTE) |
| `m_vizinhos_adasyn` | 5 | Vizinhos na base toda usados para pesar a geração (ADASYN) |
| `k_enn` | 3 | Vizinhos usados na limpeza (ENN e NCR) |
| `k_nearmiss` | 3 | Positivos mais próximos usados no NearMiss-1 |
| `max_candidatos_enn` | 20.000 | Máximo de negativos avaliados pelo ENN aproximado |
| `limite_positivos_vizinhanca` | 20.000 | Acima disso, as técnicas que processam os positivos no *driver* são excluídas |
| `n_grupos_agrupamento` | 50 | Grupos do KMeans na subamostragem por agrupamento |
| `n_folds_cross_fitting` | 3 | Partes do *cross-fitting* (IHT e negativos confiáveis) |
| `negativos_por_positivo_avaliacao_treino` | 20 | Tamanho da amostra do treino usada para medir o sobreajuste |
| `n_candidatos_bootstrap` | 10 | Configurações, além do baseline, que recebem intervalo bootstrap |
| `retomar` | `True` | Se a execução cair, a próxima pula as configurações já concluídas |
| `motor_proxy` | `"pandas"` | Onde o *proxy* treina: `"pandas"` (no *driver*) ou `"synapseml"` (distribuído) |
| `limite_memoria_proxy_mb` | 4.000 | Acima dessa estimativa de memória, a configuração usa o SynapseML |
| `proxy` | ver o código | Hiperparâmetros fixos do LightGBM do *proxy*: até 150 iterações, taxa de aprendizado 0,1, 31 folhas, parada antecipada após 30 iterações sem melhora |

O catálogo `TECNICAS` descreve cada técnica:

- `niveis`: o parâmetro com a lista de níveis testados;
- `sorteio`: se a técnica sorteia linhas, e por isso roda com várias sementes;
- `distancia`: se precisa das features padronizadas;
- `driver`: se processa os positivos no *driver*;
- `elegivel`: se pode ir para a lista curta. A técnica de pesos não pode, porque a função do Optuna não aceita coluna de peso.

In [ ]:
import json
import time
import warnings
import builtins
from datetime import datetime
import functools  # functools.reduce: o "import *" do Spark traz outra função chamada reduce

import numpy as np
import pandas as pd

from pyspark.sql import Window
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.clustering import KMeans
from pyspark.ml.functions import vector_to_array

PARAMETROS_BLOCO2 = {
    "max_features_distancia": 30,
    "k_vizinhos": 5,
    "m_vizinhos_borderline": 10,
    "m_vizinhos_adasyn": 5,
    "k_enn": 3,
    "k_nearmiss": 3,
    "max_candidatos_enn": 20000,
    "limite_positivos_vizinhanca": 20000,
    "n_grupos_agrupamento": 50,
    "max_linhas_kmeans": 500000,
    "n_folds_cross_fitting": 3,
    "negativos_por_positivo_avaliacao_treino": 20,
    "n_candidatos_bootstrap": 10,
    "tamanho_bloco_linhas": 2000,
    "tamanho_bloco_consultas": 5000,
    "retomar": True,
    "motor_proxy": "pandas",
    "limite_memoria_proxy_mb": 4000,
    "proxy": {
        "objective": "binary",
        "numIterations": 150,
        "learningRate": 0.1,
        "numLeaves": 31,
        "minDataInLeaf": 20,
        "featureFraction": 1.0,
        "baggingFraction": 1.0,
        "baggingFreq": 0,
        "lambdaL2": 1.0,
        "earlyStoppingRound": 30,
        "deterministic": True,
        "verbosity": -1,
    },
}

TECNICAS = {
    "baseline":             {"familia": "referência",      "niveis": None,                          "sorteio": False, "distancia": False, "driver": False, "elegivel": True},
    "rus":                  {"familia": "subamostragem",   "niveis": "niveis_subamostragem",        "sorteio": True,  "distancia": False, "driver": False, "elegivel": True},
    "agrupamento":          {"familia": "subamostragem",   "niveis": "niveis_subamostragem",        "sorteio": True,  "distancia": True,  "driver": False, "elegivel": True},
    "nearmiss1":            {"familia": "subamostragem",   "niveis": "niveis_subamostragem",        "sorteio": False, "distancia": True,  "driver": True,  "elegivel": True},
    "iht":                  {"familia": "subamostragem",   "niveis": "niveis_subamostragem",        "sorteio": False, "distancia": False, "driver": False, "elegivel": True},
    "tomek":                {"familia": "subamostragem",   "niveis": None,                          "sorteio": False, "distancia": True,  "driver": True,  "elegivel": True},
    "enn":                  {"familia": "subamostragem",   "niveis": None,                          "sorteio": False, "distancia": True,  "driver": True,  "elegivel": True},
    "ncr":                  {"familia": "subamostragem",   "niveis": None,                          "sorteio": False, "distancia": True,  "driver": True,  "elegivel": True},
    "negativos_confiaveis": {"familia": "subamostragem",   "niveis": "niveis_negativos_confiaveis", "sorteio": False, "distancia": False, "driver": False, "elegivel": True},
    "ros":                  {"familia": "sobreamostragem", "niveis": "niveis_sobreamostragem",      "sorteio": True,  "distancia": False, "driver": False, "elegivel": True},
    "smote":                {"familia": "sobreamostragem", "niveis": "niveis_sobreamostragem",      "sorteio": True,  "distancia": True,  "driver": True,  "elegivel": True},
    "borderline_smote":     {"familia": "sobreamostragem", "niveis": "niveis_sobreamostragem",      "sorteio": True,  "distancia": True,  "driver": True,  "elegivel": True},
    "adasyn":               {"familia": "sobreamostragem", "niveis": "niveis_sobreamostragem",      "sorteio": True,  "distancia": True,  "driver": True,  "elegivel": True},
    "rose":                 {"familia": "sobreamostragem", "niveis": "niveis_sobreamostragem",      "sorteio": True,  "distancia": False, "driver": True,  "elegivel": True},
    "smote_tomek":          {"familia": "híbrida",         "niveis": "niveis_sobreamostragem",      "sorteio": True,  "distancia": True,  "driver": True,  "elegivel": True},
    "smote_enn":            {"familia": "híbrida",         "niveis": "niveis_sobreamostragem",      "sorteio": True,  "distancia": True,  "driver": True,  "elegivel": True},
    "pesos":                {"familia": "algoritmo",       "niveis": "niveis_peso",                 "sorteio": False, "distancia": False, "driver": False, "elegivel": False},
}

TECNICAS_SMOTE = {"smote", "borderline_smote", "adasyn", "smote_tomek", "smote_enn"}


def montar_parametros_bloco2(contexto, **sobrescritos):
    """Parâmetros do bloco 1 (do contexto) + padrões do bloco 2 + sobrescritos pelo usuário."""
    conhecidos = set(PARAMETROS_PADRAO) | set(PARAMETROS_BLOCO2)
    desconhecidos = set(sobrescritos) - conhecidos
    if desconhecidos:
        raise ValueError(f"Parâmetros desconhecidos: {sorted(desconhecidos)}")
    parametros = {k: (dict(v) if isinstance(v, dict) else v) for k, v in PARAMETROS_BLOCO2.items()}
    for k, v in contexto["parametros"].items():
        if k not in parametros:
            parametros[k] = v
    for k, v in sobrescritos.items():
        if k == "proxy":
            parametros["proxy"].update(v)
        else:
            parametros[k] = v
    return parametros

## Célula 2 — Métricas

Todas as métricas são calculadas no *driver* com numpy, a partir de três vetores: alvo, score e peso. O peso permite usar amostras (a amostra de avaliação do treino) e o bootstrap. Empates de score são tratados como um único ponto de corte.

| Métrica | Cálculo |
|---|---|
| `ks` | Maior distância entre a taxa de verdadeiros positivos (TPR) e a de falsos positivos (FPR), em escala de 0 a 1 |
| `auc_roc` | Área sob a curva ROC, pela regra do trapézio. Coincide com o `areaUnderROC` do Spark |
| `gini` | 2 × AUC − 1 |
| `pr_auc` | Área sob a curva precisão × revocação, pela mesma regra do `areaUnderPR` do Spark (a curva começa no ponto revocação 0 com a precisão do primeiro corte). O Spark, por padrão, agrupa os scores em 1.000 faixas antes de calcular, o que pode mudar a terceira casa decimal |
| `revocacao_<taxa>`, `precisao_<taxa>`, `lift_<taxa>` | No corte que alerta a fração `taxa` da base com maior score. Exemplo: `revocacao_1pct` |

`bootstrap_pareado` mede a incerteza da diferença entre duas configurações. Em cada rodada, cada chave recebe um peso sorteado de uma distribuição de Poisson com média 1 (*Poisson bootstrap*: equivale a sortear chaves com reposição), e as duas configurações são medidas com os mesmos pesos. Configurações com várias sementes entram pela média entre as sementes.

In [ ]:
NOME_METRICA = {"ks": "ks", "roc": "auc_roc", "auc": "auc_roc", "pr_auc": "pr_auc"}
METRICA_PARADA = {"ks": "auc", "roc": "auc", "auc": "auc", "pr_auc": "average_precision"}


def rotulo_taxa(taxa):
    return f"{taxa * 100:g}".replace(".", "_") + "pct"


def colunas_operacionais(taxas):
    return [f"{m}_{rotulo_taxa(t)}" for t in taxas for m in ("revocacao", "precisao", "lift")]


def _trapezio(x, y):
    x = np.asarray(x, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)
    return float(np.sum(np.diff(x) * (y[1:] + y[:-1]) / 2.0))


def ordenar_scores(scores):
    """Ordem decrescente do score e o índice do último elemento de cada grupo de empate."""
    scores = np.asarray(scores, dtype=np.float64)
    ordem = np.argsort(-scores, kind="mergesort")
    ordenados = scores[ordem]
    fim = np.r_[np.nonzero(np.diff(ordenados))[0], len(ordenados) - 1]
    return ordem, fim


def metricas_de_ordem(y_ord, w_ord, fim, taxas):
    resultado = {"ks": np.nan, "auc_roc": np.nan, "gini": np.nan, "pr_auc": np.nan}
    for c in colunas_operacionais(taxas):
        resultado[c] = np.nan
    tp = np.cumsum(w_ord * y_ord)
    fp = np.cumsum(w_ord * (1.0 - y_ord))
    total_pos, total_neg = tp[-1], fp[-1]
    if total_pos <= 0 or total_neg <= 0:
        return resultado
    tp_c, fp_c = tp[fim], fp[fim]
    com_peso = (tp_c + fp_c) > 0          # cortes iniciais só com peso zero não existem na base
    tp_c, fp_c = tp_c[com_peso], fp_c[com_peso]
    tpr, fpr = tp_c / total_pos, fp_c / total_neg
    resultado["ks"] = float(np.max(np.abs(tpr - fpr)))
    auc = _trapezio(np.r_[0.0, fpr], np.r_[0.0, tpr])
    resultado["auc_roc"] = auc
    resultado["gini"] = 2.0 * auc - 1.0
    precisao = tp_c / (tp_c + fp_c)
    resultado["pr_auc"] = _trapezio(np.r_[0.0, tpr], np.r_[precisao[0], precisao])
    peso_total = total_pos + total_neg
    prevalencia = total_pos / peso_total
    acumulado = np.cumsum(w_ord)
    for taxa in taxas:
        n = int(np.searchsorted(acumulado, taxa * peso_total, side="right"))
        n = builtins.max(n, 1)
        alertas, capturados = acumulado[n - 1], tp[n - 1]
        sufixo = rotulo_taxa(taxa)
        resultado[f"revocacao_{sufixo}"] = float(capturados / total_pos)
        resultado[f"precisao_{sufixo}"] = float(capturados / alertas)
        resultado[f"lift_{sufixo}"] = float(capturados / alertas / prevalencia)
    return resultado


def calcular_metricas(y, scores, pesos=None, taxas=()):
    y = np.asarray(y, dtype=np.float64)
    pesos = np.ones_like(y) if pesos is None else np.asarray(pesos, dtype=np.float64)
    ordem, fim = ordenar_scores(scores)
    return metricas_de_ordem(y[ordem], pesos[ordem], fim, taxas)


def metricas_por_safra(y, scores, safras, metrica):
    nome = NOME_METRICA[metrica]
    y, scores, safras = np.asarray(y), np.asarray(scores), np.asarray(safras)
    saida = {}
    for safra in pd.unique(safras):
        mascara = safras == safra
        saida[str(safra)] = calcular_metricas(y[mascara], scores[mascara])[nome]
    return saida


def bootstrap_pareado(y, codigos_chave, scores_a, scores_b, metrica, n_bootstrap, semente):
    """Diferença (A - B) da métrica, com reamostragem por chave. scores_a e scores_b são listas (uma por semente)."""
    nome = NOME_METRICA[metrica]
    y = np.asarray(y, dtype=np.float64)
    codigos_chave = np.asarray(codigos_chave)
    n_chaves = int(codigos_chave.max()) + 1
    pre_a = [ordenar_scores(s) for s in scores_a]
    pre_b = [ordenar_scores(s) for s in scores_b]

    def valor(pre, pesos):
        return float(np.mean([metricas_de_ordem(y[o], pesos[o], f, ())[nome] for o, f in pre]))

    pesos_um = np.ones_like(y)
    pontual = valor(pre_a, pesos_um) - valor(pre_b, pesos_um)
    rng = np.random.default_rng(semente)
    diferencas = np.empty(n_bootstrap)
    for b in range(n_bootstrap):
        pesos = rng.poisson(1.0, n_chaves).astype(np.float64)[codigos_chave]
        diferencas[b] = valor(pre_a, pesos) - valor(pre_b, pesos)
    return {
        "diferenca": pontual,
        "ic_inferior": float(np.quantile(diferencas, 0.025)),
        "ic_superior": float(np.quantile(diferencas, 0.975)),
        "prob_ganho": float(np.mean(diferencas > 0)),
    }

## Célula 3 — *Proxy*: LightGBM em pandas, com o SynapseML como alternativa

O parâmetro `motor_proxy` escolhe onde o *proxy* treina:

| Motor | Como funciona | Quando usar |
|---|---|---|
| `"pandas"` (padrão) | O treino da técnica é convertido para pandas e treinado com a biblioteca `lightgbm` no *driver* (o nó coordenador do cluster), usando todos os seus núcleos. O teste e a amostra de avaliação do treino são convertidos uma única vez e reaproveitados em todas as configurações | Bases que cabem na memória do *driver* |
| `"synapseml"` | O `LightGBMClassifier` do SynapseML treina distribuído no cluster | Bases grandes demais para o *driver* |

Antes de cada treino em pandas, a memória é estimada como linhas × (features + 2) × 8 bytes, somando o treino da técnica e o teste. Se passar de `limite_memoria_proxy_mb`, aquela configuração usa o SynapseML, e isso fica registrado na coluna `info` dos resultados.

Nos dois motores:

- Os hiperparâmetros vêm de `parametros["proxy"]` (nomes do SynapseML; para o pandas, são traduzidos para os nomes da biblioteca Python).
- O treino usa parada antecipada (*early stopping*: para quando a métrica no teste deixa de melhorar por `earlyStoppingRound` iterações). Não há KS como métrica de parada; para `ks` e AUC-ROC é usada a AUC, e para PR-AUC a `average_precision`.
- As features são renomeadas para `f0`, `f1`, ... e convertidas para `double`, para que nomes com ponto ou espaço não causem erro.

Funções:

- `treinar_proxy`: escolhe o motor e treina.
- `prever_pandas`: score de um DataFrame pandas já no formato `f0`, `f1`, ...
- `pontuar`: score de um DataFrame Spark, nos dois motores. No motor pandas, converte para pandas se couber no limite; senão, pontua distribuído com `mapInPandas` (exige a `lightgbm` instalada também nos nós de trabalho).
- `importancias_proxy`: importância por ganho (`gain`) de cada feature.

In [ ]:
TRADUCAO_LIGHTGBM_PYTHON = {
    "objective": "objective", "learningRate": "learning_rate", "numLeaves": "num_leaves",
    "minDataInLeaf": "min_data_in_leaf", "featureFraction": "feature_fraction",
    "baggingFraction": "bagging_fraction", "baggingFreq": "bagging_freq", "lambdaL2": "lambda_l2",
    "deterministic": "deterministic", "verbosity": "verbosity",
}


def _nomes_proxy(features):
    return [f"f{i}" for i in range(len(features))]


def _matriz_spark(df, features, col_alvo=None, col_peso=None, manter=()):
    colunas = [coluna(c) for c in manter]
    if col_alvo:
        colunas.append(coluna(col_alvo).cast("double").alias("_rotulo"))
    colunas += [coluna(c).cast("double").alias(n) for c, n in zip(features, _nomes_proxy(features))]
    if col_peso:
        colunas.append(coluna(col_peso).cast("double").alias("_peso_proxy"))
    return df.select(*colunas)


def matriz_pandas(df, features, col_alvo=None, manter=()):
    """DataFrame pandas com as colunas f0, f1, ... (e _rotulo), ordenado por _id_linha quando houver."""
    pdf = _matriz_spark(df, features, col_alvo=col_alvo, manter=manter).toPandas()
    if "_id_linha" in pdf.columns:
        pdf = pdf.sort_values("_id_linha").reset_index(drop=True)
    return pdf


def _megabytes(linhas, features):
    return linhas * (len(features) + 2) * 8 / 1e6


def _aplicar_config(estimador, config):
    ignorados = []
    for nome, valor in config.items():
        if estimador.hasParam(nome):
            estimador.set(estimador.getParam(nome), valor)
        else:
            ignorados.append(nome)
    return ignorados


def treinar_proxy_synapseml(df_treino, df_validacao, features, col_alvo, metrica, semente, parametros, col_peso=None):
    from synapse.ml.lightgbm import LightGBMClassifier

    nomes = _nomes_proxy(features)

    def matriz(df, validacao):
        colunas = [coluna(col_alvo).cast("double").alias("_rotulo")]
        colunas += [coluna(c).cast("double").alias(n) for c, n in zip(features, nomes)]
        if col_peso:
            peso = F.lit(1.0) if validacao else coluna(col_peso).cast("double")
            colunas.append(peso.alias("_peso_proxy"))
        return df.select(*colunas).withColumn("_validacao", F.lit(validacao))

    dados = matriz(df_treino, False).unionByName(matriz(df_validacao, True))
    montador = VectorAssembler(inputCols=nomes, outputCol="_vetor", handleInvalid="keep")
    dados = montador.transform(dados)

    estimador = LightGBMClassifier(featuresCol="_vetor", labelCol="_rotulo", validationIndicatorCol="_validacao")
    config = dict(parametros["proxy"])
    config["metric"] = METRICA_PARADA[metrica]
    config["seed"] = int(semente)
    config["isUnbalance"] = False
    if col_peso:
        config["weightCol"] = "_peso_proxy"
    ignorados = _aplicar_config(estimador, config)
    modelo = estimador.fit(dados)
    return {"motor": "synapseml", "modelo": modelo, "montador": montador, "features": list(features),
            "nomes": nomes, "ignorados": ignorados}


def treinar_proxy_pandas(treino_pd, validacao_pd, features, metrica, semente, parametros):
    import lightgbm as lgb

    nomes = _nomes_proxy(features)
    config = parametros["proxy"]
    params = {TRADUCAO_LIGHTGBM_PYTHON[k]: v for k, v in config.items() if k in TRADUCAO_LIGHTGBM_PYTHON}
    params.update({"metric": METRICA_PARADA[metrica], "seed": int(semente), "is_unbalance": False,
                   "force_col_wise": True})
    pesos = treino_pd["_peso_proxy"].to_numpy() if "_peso_proxy" in treino_pd.columns else None
    dados_treino = lgb.Dataset(treino_pd[nomes].to_numpy(dtype=np.float64), label=treino_pd["_rotulo"].to_numpy(),
                               weight=pesos, feature_name=nomes, free_raw_data=True)
    dados_validacao = lgb.Dataset(validacao_pd[nomes].to_numpy(dtype=np.float64),
                                  label=validacao_pd["_rotulo"].to_numpy(), reference=dados_treino)
    modelo = lgb.train(params, dados_treino, num_boost_round=int(config["numIterations"]),
                       valid_sets=[dados_validacao],
                       callbacks=[lgb.early_stopping(int(config["earlyStoppingRound"]), verbose=False)])
    return {"motor": "pandas", "modelo": modelo, "features": list(features), "nomes": nomes, "ignorados": [],
            "iteracoes": int(modelo.best_iteration or modelo.current_iteration())}


def treinar_proxy(df_treino, df_validacao, features, col_alvo, metrica, semente, parametros, col_peso=None, cache=None):
    """Treina o proxy no motor escolhido. `cache` guarda a validação já em pandas entre chamadas."""
    if parametros["motor_proxy"] == "pandas":
        cache = {} if cache is None else cache
        if "validacao" not in cache:
            linhas_validacao = df_validacao.count()
            cache["validacao_mb"] = _megabytes(linhas_validacao, features)
            cache["validacao"] = (matriz_pandas(df_validacao, features, col_alvo=col_alvo)
                                  if cache["validacao_mb"] <= parametros["limite_memoria_proxy_mb"] else None)
        linhas = df_treino.count()
        memoria = _megabytes(linhas, features) + cache["validacao_mb"]
        if cache["validacao"] is not None and memoria <= parametros["limite_memoria_proxy_mb"]:
            treino_pd = _matriz_spark(df_treino, features, col_alvo=col_alvo, col_peso=col_peso).toPandas()
            proxy = treinar_proxy_pandas(treino_pd, cache["validacao"], features, metrica, semente, parametros)
            proxy["memoria_mb"] = memoria
            return proxy
        log_pipeline(f"Proxy: {memoria:,.0f} MB estimados, acima de limite_memoria_proxy_mb; usando o SynapseML.")
    return treinar_proxy_synapseml(df_treino, df_validacao, features, col_alvo, metrica, semente, parametros, col_peso)


def prever_pandas(proxy, pdf):
    modelo = proxy["modelo"]
    return modelo.predict(pdf[proxy["nomes"]].to_numpy(dtype=np.float64),
                          num_iteration=modelo.best_iteration or None)


def pontuar(proxy, df, manter, parametros=None):
    """Score de um DataFrame Spark: devolve as colunas de `manter` e `score`."""
    if proxy["motor"] == "synapseml":
        dados = proxy["montador"].transform(_matriz_spark(df, proxy["features"], manter=manter))
        saida = proxy["modelo"].transform(dados)
        return saida.select(*[coluna(c) for c in manter], vector_to_array("probability")[1].alias("score"))

    limite = (parametros or {}).get("limite_memoria_proxy_mb", PARAMETROS_BLOCO2["limite_memoria_proxy_mb"])
    matriz = _matriz_spark(df, proxy["features"], manter=manter)
    if _megabytes(df.count(), proxy["features"]) <= limite:
        pdf = matriz.toPandas()
        pdf["score"] = prever_pandas(proxy, pdf)
        return spark.createDataFrame(pdf[list(manter) + ["score"]])

    texto_modelo = proxy["modelo"].model_to_string()
    nomes = proxy["nomes"]
    esquema = T.StructType([f for f in matriz.schema.fields if f.name in manter] + [T.StructField("score", T.DoubleType())])

    def funcao(lotes):
        import lightgbm as lgb
        modelo = lgb.Booster(model_str=texto_modelo)
        for pdf in lotes:
            pdf["score"] = modelo.predict(pdf[nomes].to_numpy(dtype=np.float64), num_iteration=modelo.best_iteration or None)
            yield pdf[list(manter) + ["score"]]

    return matriz.mapInPandas(funcao, schema=esquema)


def importancias_proxy(proxy):
    try:
        if proxy["motor"] == "pandas":
            valores = list(proxy["modelo"].feature_importance("gain"))
        else:
            valores = list(proxy["modelo"].getFeatureImportances("gain"))
    except Exception:
        return {}
    return {f: float(v) for f, v in zip(proxy["features"], valores)}

## Célula 4 — Vizinhança

As técnicas de vizinhança comparam pontos pela distância euclidiana nas features padronizadas.

- **Features usadas:** as numéricas que não são categóricas codificadas; se forem mais de `max_features_distancia`, ficam as de maior importância no baseline.
- **Padronização:** nulos e NaN viram a mediana do treino; depois, cada feature tem a média subtraída e é dividida pelo desvio padrão do treino.
- `k_menores`: para cada linha de A, as k menores distâncias às linhas de B e os índices correspondentes. Trabalha em blocos para limitar a memória: `tamanho_bloco_linhas` × `tamanho_bloco_consultas` distâncias por vez (2.000 × 5.000 = 80 MB).
- **Busca no *driver*:** os positivos são poucos, então positivo × positivo é calculado no *driver*.
- **Busca distribuída** (`top_m_negativos`): os pontos de consulta vão para todos os nós dentro da própria função, e cada lote de negativos calcula as distâncias com `mapInPandas` (função do Spark que aplica código pandas a cada lote). Os m menores de cada lote são combinados no final com uma janela do Spark.
- `distancias_nearmiss`: para cada negativo, a média das distâncias aos `k_nearmiss` positivos mais próximos, também distribuída.

In [ ]:
def _padronizar(matriz, medianas, medias, desvios):
    x = np.array(matriz, dtype=np.float64, copy=True)
    faltantes = ~np.isfinite(x)
    if faltantes.any():
        x[faltantes] = np.broadcast_to(medianas, x.shape)[faltantes]
    return (x - medias) / desvios


def k_menores(A, B, k, bloco_a, bloco_b, ids_a=None, ids_b=None):
    """Para cada linha de A, as k menores distâncias euclidianas às linhas de B e os índices em B.

    Pares com o mesmo identificador (ids_a == ids_b) são ignorados, para excluir o próprio ponto.
    Posições sem vizinho ficam com distância infinita e índice -1.
    """
    n_a, n_b = len(A), len(B)
    dist = np.full((n_a, k), np.inf)
    idx = np.full((n_a, k), -1, dtype=np.int64)
    if n_a == 0 or n_b == 0:
        return dist, idx
    normas_b = np.einsum("ij,ij->i", B, B)
    for a0 in range(0, n_a, bloco_a):
        Aa = A[a0:a0 + bloco_a]
        normas_a = np.einsum("ij,ij->i", Aa, Aa)
        melhor_d = np.empty((len(Aa), 0))
        melhor_i = np.empty((len(Aa), 0), dtype=np.int64)
        for b0 in range(0, n_b, bloco_b):
            Bb = B[b0:b0 + bloco_b]
            D = normas_a[:, None] + normas_b[None, b0:b0 + len(Bb)] - 2.0 * (Aa @ Bb.T)
            np.maximum(D, 0.0, out=D)
            if ids_a is not None:
                D[ids_a[a0:a0 + len(Aa)][:, None] == ids_b[b0:b0 + len(Bb)][None, :]] = np.inf
            I = np.broadcast_to(np.arange(b0, b0 + len(Bb), dtype=np.int64), D.shape)
            D = np.concatenate([melhor_d, D], axis=1)
            I = np.concatenate([melhor_i, I], axis=1)
            if D.shape[1] > k:
                sel = np.argpartition(D, k - 1, axis=1)[:, :k]
                D = np.take_along_axis(D, sel, axis=1)
                I = np.take_along_axis(I, sel, axis=1)
            melhor_d, melhor_i = D, I
        ordem = np.argsort(melhor_d, axis=1, kind="stable")
        melhor_d = np.take_along_axis(melhor_d, ordem, axis=1)
        melhor_i = np.take_along_axis(melhor_i, ordem, axis=1)
        n_col = melhor_d.shape[1]
        dist[a0:a0 + len(Aa), :n_col] = np.sqrt(melhor_d)
        idx[a0:a0 + len(Aa), :n_col] = np.where(np.isfinite(melhor_d), melhor_i, -1)
    return dist, idx


def _config_padronizacao(base):
    return (base["nomes_dist"], base["medianas"], base["medias"], base["desvios"])


def top_m_negativos(base, consultas, m, ids_consultas=None):
    """Para cada linha de `consultas` (já padronizada), os m negativos do treino mais próximos.

    Devolve (distâncias, _id_linha dos negativos), ambos com forma (n_consultas, m).
    Com `ids_consultas`, um negativo não é considerado vizinho de si mesmo.
    """
    nomes, medianas, medias, desvios = _config_padronizacao(base)
    bloco_linhas = base["parametros"]["tamanho_bloco_linhas"]
    bloco_consultas = base["parametros"]["tamanho_bloco_consultas"]
    Q = np.ascontiguousarray(consultas, dtype=np.float64)
    ids_q = None if ids_consultas is None else np.asarray(ids_consultas, dtype=np.int64)

    def funcao(lotes):
        for pdf in lotes:
            if len(pdf) == 0:
                continue
            X = _padronizar(pdf[nomes].to_numpy(dtype=np.float64), medianas, medias, desvios)
            ids = pdf["_id_linha"].to_numpy(dtype=np.int64)
            d, i = k_menores(Q, X, m, bloco_consultas, bloco_linhas,
                             ids_a=ids_q, ids_b=ids if ids_q is not None else None)
            linhas, colunas_ok = np.nonzero(i >= 0)
            yield pd.DataFrame({
                "consulta": linhas.astype(np.int32),
                "_id_linha": ids[i[linhas, colunas_ok]],
                "dist": d[linhas, colunas_ok],
            })

    saida = base["neg_dist"].mapInPandas(funcao, schema="consulta int, _id_linha long, dist double")
    janela = Window.partitionBy("consulta").orderBy("dist", "_id_linha")
    top = (saida.withColumn("_ordem", F.row_number().over(janela))
                .filter(F.col("_ordem") <= m)
                .toPandas())
    dist = np.full((len(Q), m), np.inf)
    ids_viz = np.full((len(Q), m), -1, dtype=np.int64)
    if len(top):
        lin = top["consulta"].to_numpy()
        col = top["_ordem"].to_numpy() - 1
        dist[lin, col] = top["dist"].to_numpy()
        ids_viz[lin, col] = top["_id_linha"].to_numpy()
    return dist, ids_viz


def distancias_nearmiss(base, k):
    """Para cada negativo: média das distâncias aos k positivos mais próximos (NearMiss-1)."""
    nomes, medianas, medias, desvios = _config_padronizacao(base)
    bloco_linhas = base["parametros"]["tamanho_bloco_linhas"]
    bloco_consultas = base["parametros"]["tamanho_bloco_consultas"]
    P = np.ascontiguousarray(base["P"])

    def funcao(lotes):
        for pdf in lotes:
            if len(pdf) == 0:
                continue
            X = _padronizar(pdf[nomes].to_numpy(dtype=np.float64), medianas, medias, desvios)
            d, _ = k_menores(X, P, k, bloco_linhas, bloco_consultas)
            yield pd.DataFrame({"_id_linha": pdf["_id_linha"].to_numpy(dtype=np.int64), "nm": d.mean(axis=1)})

    return base["neg_dist"].mapInPandas(funcao, schema="_id_linha long, nm double")


def matriz_negativos(base, ids):
    """Coleta e padroniza os negativos com os `_id_linha` informados, na mesma ordem."""
    nomes, medianas, medias, desvios = _config_padronizacao(base)
    ids = np.asarray(ids, dtype=np.int64)
    if len(ids) == 0:
        return np.empty((0, len(nomes)))
    ids_df = spark.createDataFrame(pd.DataFrame({"_id_linha": ids}))
    pdf = base["neg_dist"].join(ids_df, on="_id_linha", how="inner").toPandas().set_index("_id_linha").loc[ids]
    return _padronizar(pdf[nomes].to_numpy(dtype=np.float64), medianas, medias, desvios)


def estatisticas_padronizacao(df, features_dist):
    agregacoes = []
    for i, c in enumerate(features_dist):
        x = coluna(c).cast("double")
        x = F.when(~F.isnan(x), x)
        agregacoes += [F.percentile_approx(x, 0.5, 10000).alias(f"m{i}"),
                       F.avg(x).alias(f"u{i}"), F.stddev(x).alias(f"s{i}")]
    r = df.agg(*agregacoes).first()
    medianas = np.array([r[f"m{i}"] if r[f"m{i}"] is not None else 0.0 for i in range(len(features_dist))], dtype=np.float64)
    medias = np.array([r[f"u{i}"] if r[f"u{i}"] is not None else 0.0 for i in range(len(features_dist))], dtype=np.float64)
    desvios = np.array([r[f"s{i}"] if r[f"s{i}"] not in (None, 0) else 1.0 for i in range(len(features_dist))], dtype=np.float64)
    desvios[~np.isfinite(desvios) | (desvios == 0)] = 1.0
    return medianas, medias, desvios


def escolher_features_distancia(features, categoricas, importancias, maximo):
    candidatas = [f for f in features if f not in set(categoricas)]
    if len(candidatas) <= maximo:
        return candidatas
    return sorted(candidatas, key=lambda f: -importancias.get(f, 0.0))[:maximo]


def preparar_vizinhanca(base, features_dist):
    """Padronização, coleta dos positivos e vizinhos dos positivos (entre si e entre os negativos)."""
    p = base["parametros"]
    alvo = base["alvo"]
    base["features_dist"] = list(features_dist)
    base["nomes_dist"] = [f"d{i}" for i in range(len(features_dist))]
    base["medianas"], base["medias"], base["desvios"] = estatisticas_padronizacao(base["treino"], features_dist)
    base["neg_dist"] = (base["treino"].filter(coluna(alvo) == 0)
                        .select(F.col("_id_linha"),
                                *[coluna(c).cast("double").alias(n) for c, n in zip(features_dist, base["nomes_dist"])]))
    if base["n_pos"] > p["limite_positivos_vizinhanca"]:
        base["vizinhanca_ok"] = False
        return base
    pos = (base["treino"].filter(coluna(alvo) == 1)
           .select(F.col("_id_linha"), coluna(base["safra"]),
                   *[coluna(c).cast("double").alias(c) for c in base["features"]])
           .orderBy("_id_linha")
           .toPandas())
    base["positivos"] = pos
    base["X_pos"] = pos[base["features"]].to_numpy(dtype=np.float64)
    base["P"] = _padronizar(pos[features_dist].to_numpy(dtype=np.float64), base["medianas"], base["medias"], base["desvios"])
    base["k_max"] = builtins.max(p["k_vizinhos"], p["m_vizinhos_borderline"], p["m_vizinhos_adasyn"], p["k_enn"])
    sequencia = np.arange(len(pos), dtype=np.int64)
    log_pipeline(f"Vizinhança: {len(pos):,} positivos, {len(features_dist)} features de distância.")
    base["viz_pp"] = k_menores(base["P"], base["P"], base["k_max"], p["tamanho_bloco_consultas"],
                               p["tamanho_bloco_linhas"], ids_a=sequencia, ids_b=sequencia)
    base["viz_pn"] = top_m_negativos(base, base["P"], base["k_max"])
    base["vizinhanca_ok"] = True
    return base

## Célula 5 — Análises compartilhadas

São calculadas uma vez e reaproveitadas por todos os níveis e sementes.

| Análise | Regra |
|---|---|
| **Vizinhos gerais de um positivo** | Junta os vizinhos positivos e negativos e fica com os k mais próximos no total |
| **Tomek links** (Tomek, 1976) | Par positivo-negativo em que cada um é o vizinho mais próximo do outro. O negativo é removido. Exato, em duas passadas distribuídas |
| **ENN aproximado** (Wilson, 1972) | Remove o negativo cuja maioria dos `k_enn` vizinhos é positiva. Aproximação: só são avaliados os negativos que estão entre os `k_enn` vizinhos de algum positivo, até `max_candidatos_enn` |
| **NCR aproximado** (Laurikkala, 2001) | ENN mais a remoção dos vizinhos negativos de cada positivo que os seus `k_enn` vizinhos classificariam como negativo |
| **NearMiss-1** (Mani e Zhang, 2003) | Média das distâncias de cada negativo aos `k_nearmiss` positivos mais próximos; ficam os de menor média |
| **Agrupamento** | KMeans com `n_grupos_agrupamento` grupos sobre os negativos padronizados (ajustado em até `max_linhas_kmeans` linhas) |
| ***Cross-fitting*** | O treino é dividido em `n_folds_cross_fitting` partes por chave; cada parte é pontuada por um *proxy* treinado nas demais. Assim, o score de um negativo não vem de um modelo que já o viu. Usado pelo IHT e pelos negativos confiáveis |

As análises mais caras (NearMiss, agrupamento e *cross-fitting*) são gravadas como tabelas e reaproveitadas quando a execução é retomada.

In [ ]:
def vizinhos_gerais(viz_pp, viz_pn, k):
    """Os k vizinhos mais próximos no total: distâncias, rótulos (1 = positivo) e identificadores.

    Para vizinhos positivos o identificador é o índice do positivo; para negativos, o `_id_linha`.
    """
    d = np.concatenate([viz_pp[0][:, :k], viz_pn[0][:, :k]], axis=1)
    rotulos = np.concatenate([np.ones_like(viz_pp[0][:, :k]), np.zeros_like(viz_pn[0][:, :k])], axis=1)
    ids = np.concatenate([viz_pp[1][:, :k], viz_pn[1][:, :k]], axis=1)
    ordem = np.argsort(d, axis=1, kind="stable")[:, :k]
    return (np.take_along_axis(d, ordem, axis=1), np.take_along_axis(rotulos, ordem, axis=1),
            np.take_along_axis(ids, ordem, axis=1))


def remover_tomek(base, P, viz_pp, viz_pn):
    """`_id_linha` dos negativos que formam Tomek links com os positivos em P."""
    p = base["parametros"]
    d_pos, d_neg, j = viz_pp[0][:, 0], viz_pn[0][:, 0], viz_pn[1][:, 0]
    candidatos = (d_neg < d_pos) & (j >= 0)
    if not candidatos.any():
        return np.array([], dtype=np.int64)
    pares = (pd.DataFrame({"pos": np.nonzero(candidatos)[0], "neg": j[candidatos], "delta": d_neg[candidatos]})
               .sort_values(["neg", "delta"]).drop_duplicates("neg"))
    negs = pares["neg"].to_numpy(dtype=np.int64)
    delta = pares["delta"].to_numpy()
    J = matriz_negativos(base, negs)
    d_jp, _ = k_menores(J, P, 1, p["tamanho_bloco_consultas"], p["tamanho_bloco_linhas"])
    d_jn, _ = top_m_negativos(base, J, 1, ids_consultas=negs)
    positivo_mais_proximo = np.isclose(d_jp[:, 0], delta, rtol=1e-9, atol=1e-12)
    nenhum_negativo_mais_proximo = delta < d_jn[:, 0]
    return negs[positivo_mais_proximo & nenhum_negativo_mais_proximo]


def remover_enn(base, P, viz_pn, k):
    """ENN aproximado. Devolve (ids removidos, número de candidatos avaliados)."""
    p = base["parametros"]
    d, ids = viz_pn[0][:, :k], viz_pn[1][:, :k]
    cand = pd.DataFrame({"neg": ids.ravel(), "dist": d.ravel()})
    cand = (cand[cand["neg"] >= 0].groupby("neg", as_index=False)["dist"].min()
            .sort_values(["dist", "neg"]).head(p["max_candidatos_enn"]))
    negs = cand["neg"].to_numpy(dtype=np.int64)
    if len(negs) == 0:
        return np.array([], dtype=np.int64), 0
    J = matriz_negativos(base, negs)
    d_p, _ = k_menores(J, P, k, p["tamanho_bloco_consultas"], p["tamanho_bloco_linhas"])
    d_n, _ = top_m_negativos(base, J, k, ids_consultas=negs)
    D = np.concatenate([d_p, d_n], axis=1)
    rotulos = np.concatenate([np.ones_like(d_p), np.zeros_like(d_n)], axis=1)
    ordem = np.argsort(D, axis=1, kind="stable")[:, :k]
    positivos_entre_vizinhos = np.take_along_axis(rotulos, ordem, axis=1).sum(axis=1)
    return negs[positivos_entre_vizinhos >= k // 2 + 1], len(negs)


def remover_ncr(base, P, viz_pp, viz_pn, k):
    removidos_enn, _ = remover_enn(base, P, viz_pn, k)
    _, rotulos, ids = vizinhos_gerais(viz_pp, viz_pn, k)
    mal_classificados = (rotulos == 0).sum(axis=1) >= k // 2 + 1
    vizinhos_negativos = ids[mal_classificados][rotulos[mal_classificados] == 0]
    return np.union1d(removidos_enn, vizinhos_negativos.astype(np.int64))


def _tabela_reaproveitavel(base, nome, funcao_df):
    tabela = nome_tabela(base["destino"], base["prefixo"], nome)
    if base["reaproveitar"]:
        try:
            df = spark.table(tabela)
            df.schema
            log_pipeline(f"Reaproveitando a tabela {tabela}.")
            return df
        except Exception:
            pass
    return salvar_tabela(funcao_df(), base["destino"], base["prefixo"], nome, base["registro"])


def calcular_grupos(base):
    p = base["parametros"]
    nomes = base["nomes_dist"]
    expressoes = []
    for i, n in enumerate(nomes):
        x = F.col(n)
        x = F.when(x.isNull() | F.isnan(x), F.lit(float(base["medianas"][i]))).otherwise(x)
        expressoes.append(((x - F.lit(float(base["medias"][i]))) / F.lit(float(base["desvios"][i]))).alias(f"z{i}"))
    neg = base["neg_dist"].select(F.col("_id_linha"), *expressoes)
    neg = VectorAssembler(inputCols=[f"z{i}" for i in range(len(nomes))], outputCol="_z").transform(neg)
    n_neg = base["n_neg"]
    ajuste = neg
    if n_neg > p["max_linhas_kmeans"]:
        ajuste = neg.sample(withReplacement=False, fraction=p["max_linhas_kmeans"] / n_neg, seed=p["semente"])
    k = int(builtins.min(p["n_grupos_agrupamento"], n_neg))
    modelo = KMeans(k=k, seed=p["semente"], featuresCol="_z", predictionCol="grupo").fit(ajuste)
    return modelo.transform(neg).select("_id_linha", F.col("grupo").cast("int").alias("grupo"))


def calcular_cross_fitting(base, metrica):
    p = base["parametros"]
    n_folds = p["n_folds_cross_fitting"]
    dobra = F.floor(uniforme_por_chave(base["chave"], p["semente"] + 17) * F.lit(n_folds)).cast("int")
    treino = base["treino"].withColumn("_dobra", dobra)
    partes = []
    for f in range(n_folds):
        log_pipeline(f"Cross-fitting: parte {f + 1}/{n_folds}.")
        proxy = treinar_proxy(treino.filter(F.col("_dobra") != f), base["teste"], base["features"],
                              base["alvo"], metrica, p["semente"], p, cache=base["cache_pandas"])
        alvo_parte = treino.filter((F.col("_dobra") == f) & (coluna(base["alvo"]) == 0))
        partes.append(salvar_tabela(pontuar(proxy, alvo_parte, ["_id_linha"], p), base["destino"], base["prefixo"],
                                    f"analise_cross_fitting_parte{f}", base["registro"]))
    return functools.reduce(lambda a, b: a.unionByName(b), partes)


def obter(base, chave, funcao):
    if chave not in base["cache"]:
        base["cache"][chave] = funcao()
    return base["cache"][chave]


def analise(base, nome, metrica):
    p = base["parametros"]
    if nome == "nearmiss":
        return obter(base, nome, lambda: _tabela_reaproveitavel(
            base, "analise_nearmiss", lambda: distancias_nearmiss(base, p["k_nearmiss"])))
    if nome == "grupos":
        def calcular():
            df = _tabela_reaproveitavel(base, "analise_grupos", lambda: calcular_grupos(base))
            tamanhos = {int(r["grupo"]): int(r["count"]) for r in df.groupBy("grupo").count().collect()}
            return df, tamanhos
        return obter(base, nome, calcular)
    if nome == "cross_fitting":
        return obter(base, nome, lambda: _tabela_reaproveitavel(
            base, "analise_cross_fitting", lambda: calcular_cross_fitting(base, metrica)))
    if nome == "tomek":
        return obter(base, nome, lambda: remover_tomek(base, base["P"], base["viz_pp"], base["viz_pn"]))
    if nome == "enn":
        return obter(base, nome, lambda: remover_enn(base, base["P"], base["viz_pn"], p["k_enn"])[0])
    if nome == "ncr":
        return obter(base, nome, lambda: remover_ncr(base, base["P"], base["viz_pp"], base["viz_pn"], p["k_enn"]))
    raise ValueError(f"Análise desconhecida: {nome}")

## Célula 6 — Bases da triagem

`preparar_tabelas_triagem` grava três tabelas no destino do bloco 1:

| Tabela | Conteúdo |
|---|---|
| `base_treino` | O treino do bloco 1 com a coluna `_id_linha` (identificador fixo de cada linha) |
| `base_teste` | O teste do bloco 1 com `_id_linha` |
| `amostra_avaliacao_treino` | Todos os positivos do treino e `negativos_por_positivo_avaliacao_treino` negativos por positivo, com o peso `_peso_avaliacao` que devolve a proporção original. Serve para medir o desempenho no treino original e, com ele, o sobreajuste (diferença treino − teste) |

Gravar as tabelas fixa o `_id_linha`: sem isso, o Spark poderia recalcular o identificador de outra forma entre uma configuração e outra.

Com `retomar=True`, as tabelas existentes são reaproveitadas se tiverem as mesmas colunas e o mesmo número de linhas e de positivos do `contexto` atual.

In [ ]:
def preparar_tabelas_triagem(contexto, parametros, metrica):
    col = contexto["colunas"]
    chave, safra, alvo = col["chave"], col["safra"], col["alvo"]
    features = contexto["features_selecionadas"]
    destino, prefixo = contexto["destino_delta"], parametros["prefixo_tabelas"]
    registro = contexto["registro"]
    partes = contexto["partes"]
    esperado = [chave, safra, alvo] + features + ["_id_linha"]

    contagem_treino = calcular_prevalencia(partes["treino"], alvo)
    contagem_teste = calcular_prevalencia(partes["teste"], alvo)

    reaproveitar = False
    if parametros["retomar"]:
        try:
            tr = spark.table(nome_tabela(destino, prefixo, "base_treino"))
            te = spark.table(nome_tabela(destino, prefixo, "base_teste"))
            am = spark.table(nome_tabela(destino, prefixo, "amostra_avaliacao_treino"))
            reaproveitar = (tr.columns == esperado and te.columns == esperado
                            and calcular_prevalencia(tr, alvo) == contagem_treino
                            and calcular_prevalencia(te, alvo) == contagem_teste)
        except Exception:
            reaproveitar = False

    if reaproveitar:
        log_pipeline("Retomando: reaproveitando base_treino, base_teste e amostra_avaliacao_treino.")
    else:
        tr = salvar_tabela(partes["treino"].withColumn("_id_linha", F.monotonically_increasing_id()),
                           destino, prefixo, "base_treino", registro)
        te = salvar_tabela(partes["teste"].withColumn("_id_linha", F.monotonically_increasing_id()),
                           destino, prefixo, "base_teste", registro)
        fracao = builtins.min(1.0, parametros["negativos_por_positivo_avaliacao_treino"]
                              * contagem_treino["positivos"] / builtins.max(contagem_treino["negativos"], 1))
        amostra = (tr.filter(coluna(alvo) == 1)
                   .unionByName(tr.filter(coluna(alvo) == 0)
                                .sample(withReplacement=False, fraction=fracao, seed=parametros["semente"]))
                   .withColumn("_peso_avaliacao", F.when(coluna(alvo) == 1, F.lit(1.0)).otherwise(F.lit(1.0 / fracao))))
        am = salvar_tabela(amostra, destino, prefixo, "amostra_avaliacao_treino", registro)

    teste_info = (te.select("_id_linha", coluna(chave), coluna(safra), coluna(alvo))
                    .toPandas().sort_values("_id_linha").reset_index(drop=True))
    amostra_info = (am.select("_id_linha", coluna(alvo), "_peso_avaliacao")
                      .toPandas().sort_values("_id_linha").reset_index(drop=True))
    safras = [r[0] for r in tr.select(coluna(safra)).distinct().collect() if r[0] is not None]

    return {
        "parametros": parametros, "registro": registro, "destino": destino, "prefixo": prefixo,
        "chave": chave, "safra": safra, "alvo": alvo, "features": features,
        "categoricas": list(contexto.get("categoricas", [])),
        "treino": tr, "teste": te, "amostra_treino": am, "safras": safras,
        "n_pos": contagem_treino["positivos"], "n_neg": contagem_treino["negativos"],
        "teste_ids": teste_info["_id_linha"].to_numpy(),
        "y_teste": teste_info[alvo].to_numpy(dtype=np.float64),
        "safra_teste": teste_info[safra].astype(str).to_numpy(),
        "codigos_chave_teste": pd.factorize(teste_info[chave])[0],
        "amostra_ids": amostra_info["_id_linha"].to_numpy(),
        "y_amostra": amostra_info[alvo].to_numpy(dtype=np.float64),
        "w_amostra": amostra_info["_peso_avaliacao"].to_numpy(dtype=np.float64),
        "reaproveitar": reaproveitar, "cache": {}, "cache_pandas": {}, "vizinhanca_ok": False,
    }

## Célula 7 — As 16 técnicas

Cada técnica recebe a base e uma configuração (nível e semente) e devolve um dicionário com:

- `df`: o treino transformado, com as colunas chave, safra, alvo e features (e `_peso`, só na técnica de pesos);
- `n_pos`, `n_neg`, `n_sinteticos`: contagens do treino resultante (`n_pos` já considera o peso);
- `beta`: fator da correção de calibração, (negativos mantidos ÷ negativos originais) ÷ (positivos efetivos ÷ positivos originais).

Se a técnica não puder ser aplicada naquela base (por exemplo, nenhum positivo de fronteira para o Borderline-SMOTE), devolve `{"inaplicavel": motivo}`.

**Subamostragem** (nível = negativos por positivo, exceto nos negativos confiáveis):

| Técnica | Regra |
|---|---|
| `rus` | Sorteio de negativos estratificado por safra |
| `agrupamento` | Mesma quantidade de negativos sorteada em cada grupo do KMeans, para preservar a diversidade (adaptação do Cluster Centroids) |
| `nearmiss1` | Ficam os negativos mais próximos dos positivos |
| `iht` | *Instance Hardness Threshold* (limiar de dificuldade da instância): ficam os negativos com menor score no *cross-fitting* |
| `tomek`, `enn`, `ncr` | Remoção pela análise correspondente (sem nível) |
| `negativos_confiaveis` | Remove a fração (nível) dos negativos com maior score no *cross-fitting* |

**Sobreamostragem** (nível = multiplicador dos positivos; 5 significa que os positivos passam a ser 5 vezes o original):

| Técnica | Regra |
|---|---|
| `ros` | Réplicas dos positivos, sorteadas com reposição |
| `smote` | Ponto sintético no segmento entre um positivo e um dos `k_vizinhos` positivos mais próximos |
| `borderline_smote` | Igual, mas só a partir de positivos de fronteira: metade ou mais dos `m_vizinhos_borderline` vizinhos são negativos, mas não todos |
| `adasyn` | A chance de um positivo gerar pontos é proporcional à fração de negativos entre seus `m_vizinhos_adasyn` vizinhos |
| `rose` | Positivo sorteado mais ruído gaussiano, com largura h × desvio de cada feature, h = (4 / ((d + 2) n))^(1/(d + 4)) (Menardi e Torelli, 2014) |

Nas técnicas do tipo SMOTE, as categóricas codificadas recebem o valor mais frequente entre o positivo de origem e seus vizinhos, em vez de uma interpolação. Valores ausentes no vizinho são substituídos pelo valor do ponto de origem. Features do tipo inteiro são arredondadas.

**Híbridas:** `smote_tomek` e `smote_enn` aplicam o SMOTE e depois a limpeza, recalculando a vizinhança com os positivos sintéticos. Só negativos são removidos.

**Algoritmo:** `pesos` mantém o treino e acrescenta a coluna `_peso` (nível nos positivos, 1 nos negativos).

In [ ]:
def _colunas_saida(base):
    return [base["chave"], base["safra"], base["alvo"]] + base["features"]


def _positivos(base):
    return base["treino"].filter(coluna(base["alvo"]) == 1)


def _negativos(base):
    return base["treino"].filter(coluna(base["alvo"]) == 0)


def _sem_ids(df, ids):
    ids = np.asarray(ids, dtype=np.int64)
    if len(ids) == 0:
        return df
    ids_df = spark.createDataFrame(pd.DataFrame({"_id_linha": ids}))
    return df.join(ids_df, on="_id_linha", how="left_anti")


def _com_ids(df, ids_df):
    return df.join(ids_df.select("_id_linha"), on="_id_linha", how="left_semi")


def _montar(base, negativos, extras=None, n_extras=0, sinteticos=0):
    colunas = [coluna(c) for c in _colunas_saida(base)]
    negativos = negativos.select(*colunas).persist()
    n_neg = negativos.count()
    df = _positivos(base).select(*colunas).unionByName(negativos)
    if extras is not None:
        df = df.unionByName(extras.select(*colunas))
    n_pos = base["n_pos"] + n_extras
    beta = (n_neg / base["n_neg"]) / (n_pos / base["n_pos"])
    return {"df": df, "n_pos": float(n_pos), "n_neg": int(n_neg), "n_sinteticos": int(sinteticos),
            "beta": float(beta), "col_peso": None, "_cache": [negativos]}


def _n_manter(base, razao):
    return int(builtins.round(razao * base["n_pos"]))


def _moda_por_linha(V):
    iguais = (V[:, :, None] == V[:, None, :]).sum(axis=2)
    escolha = np.argmax(iguais, axis=1)
    return V[np.arange(len(V)), escolha]


def gerar_sinteticos(base, n_novos, semente, modo, probabilidades=None):
    """Positivos sintéticos na escala original. Devolve (pandas com safra + features, matriz padronizada)."""
    p = base["parametros"]
    rng = np.random.default_rng(semente)
    X = base["X_pos"]
    n = len(X)
    features = base["features"]
    cat = set(base["categoricas"])
    idx_cat = [i for i, f in enumerate(features) if f in cat]
    idx_cont = [i for i, f in enumerate(features) if f not in cat]
    origem = rng.integers(0, n, n_novos) if probabilidades is None else rng.choice(n, n_novos, p=probabilidades)

    if modo == "rose":
        novo = X[origem].copy()
        if idx_cont:
            d = len(idx_cont)
            h = (4.0 / ((d + 2.0) * n)) ** (1.0 / (d + 4.0))
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                desvio = np.nanstd(X[:, idx_cont], axis=0)
            desvio = np.where(np.isfinite(desvio), desvio, 0.0)
            novo[:, idx_cont] += rng.standard_normal((n_novos, d)) * (h * desvio)
    else:
        k = p["k_vizinhos"]
        vizinhos = base["viz_pp"][1][:, :k]
        escolhido = vizinhos[origem, rng.integers(0, k, n_novos)]
        lam = rng.random(n_novos)[:, None]
        A, B = X[origem], X[escolhido]
        B = np.where(np.isnan(B), A, B)
        novo = A + lam * (B - A)
        if idx_cat:
            grupo = np.concatenate([origem[:, None], vizinhos[origem]], axis=1)
            for j in idx_cat:
                novo[:, j] = _moda_por_linha(X[grupo, j])

    pdf = pd.DataFrame(novo, columns=features)
    pdf.insert(0, base["safra"], base["positivos"][base["safra"]].to_numpy()[origem])
    S = _padronizar(pdf[base["features_dist"]].to_numpy(dtype=np.float64), base["medianas"], base["medias"], base["desvios"])
    return pdf, S


def _sinteticos_para_spark(base, pdf):
    tipos = {f.name: f.dataType for f in base["treino"].schema.fields}
    sdf = spark.createDataFrame(pdf)
    colunas = [F.lit(None).cast(tipos[base["chave"]]).alias(base["chave"]),
               coluna(base["safra"]).cast(tipos[base["safra"]]).alias(base["safra"]),
               F.lit(1).cast(tipos[base["alvo"]]).alias(base["alvo"])]
    inteiros = (T.ByteType, T.ShortType, T.IntegerType, T.LongType)
    for c in base["features"]:
        x = coluna(c).cast("double")
        x = F.when(~F.isnan(x), x)
        tipo = tipos[c]
        if isinstance(tipo, inteiros):
            x = F.round(x).cast(tipo)
        elif isinstance(tipo, T.BooleanType):
            x = x >= F.lit(0.5)
        else:
            x = x.cast(tipo)
        colunas.append(x.alias(c))
    return sdf.select(*colunas)


def _probabilidades_borderline(base):
    m = base["parametros"]["m_vizinhos_borderline"]
    _, rotulos, _ = vizinhos_gerais(base["viz_pp"], base["viz_pn"], m)
    n_neg = (rotulos == 0).sum(axis=1)
    perigo = (n_neg >= m / 2.0) & (n_neg < m)
    if not perigo.any():
        return None
    return perigo / perigo.sum()


def _probabilidades_adasyn(base):
    m = base["parametros"]["m_vizinhos_adasyn"]
    _, rotulos, _ = vizinhos_gerais(base["viz_pp"], base["viz_pn"], m)
    r = (rotulos == 0).sum(axis=1) / m
    if r.sum() == 0:
        return None
    return r / r.sum()


def _hibrida(base, multiplicador, semente, limpeza):
    p = base["parametros"]
    n_novos = int(builtins.round((multiplicador - 1) * base["n_pos"]))
    pdf, S = gerar_sinteticos(base, n_novos, semente, "smote")
    P_aum = np.vstack([base["P"], S])
    sequencia = np.arange(len(P_aum), dtype=np.int64)
    k = builtins.max(1, p["k_enn"])
    viz_pp = k_menores(P_aum, P_aum, k, p["tamanho_bloco_consultas"], p["tamanho_bloco_linhas"], ids_a=sequencia, ids_b=sequencia)
    viz_pn = top_m_negativos(base, P_aum, k)
    if limpeza == "tomek":
        remover = remover_tomek(base, P_aum, viz_pp, viz_pn)
    else:
        remover, _ = remover_enn(base, P_aum, viz_pn, p["k_enn"])
    resultado = _montar(base, _sem_ids(_negativos(base), remover), _sinteticos_para_spark(base, pdf),
                        n_extras=n_novos, sinteticos=n_novos)
    resultado["removidos"] = int(len(remover))
    return resultado


def aplicar_tecnica(base, tecnica, nivel, semente, metrica):
    p = base["parametros"]
    alvo = base["alvo"]

    if tecnica == "baseline":
        return _montar(base, _negativos(base))

    if tecnica == "pesos":
        df = base["treino"].select(*[coluna(c) for c in _colunas_saida(base)]).withColumn(
            "_peso", F.when(coluna(alvo) == 1, F.lit(float(nivel))).otherwise(F.lit(1.0)))
        return {"df": df, "n_pos": float(base["n_pos"] * nivel), "n_neg": base["n_neg"], "n_sinteticos": 0,
                "beta": 1.0 / float(nivel), "col_peso": "_peso", "_cache": []}

    if tecnica == "rus":
        fracao = builtins.min(1.0, _n_manter(base, nivel) / base["n_neg"])
        neg = _negativos(base).sampleBy(base["safra"], {s: fracao for s in base["safras"]}, seed=int(semente))
        return _montar(base, neg)

    if tecnica == "agrupamento":
        grupos, tamanhos = analise(base, "grupos", metrica)
        por_grupo = _n_manter(base, nivel) / builtins.max(len(tamanhos), 1)
        fracoes = {g: builtins.min(1.0, por_grupo / t) for g, t in tamanhos.items()}
        neg = _negativos(base).join(grupos, on="_id_linha", how="inner").sampleBy("grupo", fracoes, seed=int(semente))
        return _montar(base, neg.drop("grupo"))

    if tecnica == "nearmiss1":
        ids = analise(base, "nearmiss", metrica).orderBy("nm", "_id_linha").limit(_n_manter(base, nivel))
        return _montar(base, _com_ids(_negativos(base), ids))

    if tecnica == "iht":
        scores = analise(base, "cross_fitting", metrica)
        ids = scores.orderBy("score", "_id_linha").limit(_n_manter(base, nivel))
        return _montar(base, _com_ids(_negativos(base), ids))

    if tecnica == "negativos_confiaveis":
        scores = analise(base, "cross_fitting", metrica)
        n_remover = int(builtins.round(nivel * base["n_neg"]))
        ids = scores.orderBy(F.col("score").desc(), "_id_linha").limit(n_remover)
        return _montar(base, _negativos(base).join(ids.select("_id_linha"), on="_id_linha", how="left_anti"))

    if tecnica in ("tomek", "enn", "ncr"):
        remover = analise(base, tecnica, metrica)
        resultado = _montar(base, _sem_ids(_negativos(base), remover))
        resultado["removidos"] = int(len(remover))
        return resultado

    if tecnica == "ros":
        extras = _positivos(base).sample(withReplacement=True, fraction=float(nivel - 1), seed=int(semente)).persist()
        n_extras = extras.count()
        resultado = _montar(base, _negativos(base), extras, n_extras=n_extras)
        resultado["_cache"].append(extras)
        return resultado

    if tecnica in ("smote", "borderline_smote", "adasyn", "rose"):
        n_novos = int(builtins.round((nivel - 1) * base["n_pos"]))
        probabilidades = None
        if tecnica == "borderline_smote":
            probabilidades = obter(base, "prob_borderline", lambda: _probabilidades_borderline(base))
            if probabilidades is None:
                return {"inaplicavel": "Nenhum positivo de fronteira: nenhum positivo tem entre metade e todos os vizinhos negativos."}
        if tecnica == "adasyn":
            probabilidades = obter(base, "prob_adasyn", lambda: _probabilidades_adasyn(base))
            if probabilidades is None:
                return {"inaplicavel": "Nenhum positivo tem vizinhos negativos."}
        pdf, _ = gerar_sinteticos(base, n_novos, semente, "rose" if tecnica == "rose" else "smote", probabilidades)
        return _montar(base, _negativos(base), _sinteticos_para_spark(base, pdf), n_extras=n_novos, sinteticos=n_novos)

    if tecnica == "smote_tomek":
        return _hibrida(base, nivel, semente, "tomek")
    if tecnica == "smote_enn":
        return _hibrida(base, nivel, semente, "enn")

    raise ValueError(f"Técnica desconhecida: {tecnica}")


def liberar_resultado(resultado):
    for df in resultado.get("_cache", []):
        df.unpersist()

## Célula 8 — Triagem

`executar_triagem(contexto, **parametros)` executa, nesta ordem:

1. **Bases:** grava ou reaproveita as tabelas da célula 6.
2. **Baseline:** treina o *proxy* sem técnica. As importâncias dele escolhem as features de distância.
3. **Aplicabilidade:** exclui técnicas e níveis que não fazem sentido na base, com o motivo (por exemplo, nível de subamostragem que manteria todos os negativos).
4. **Grade:** técnicas aplicáveis × níveis × sementes (só as técnicas com sorteio usam várias sementes). Com os padrões, 127 treinos.
5. **Execução:** para cada configuração, aplica a técnica, treina o *proxy* com a semente fixa do pipeline, pontua o teste e a amostra do treino e grava uma linha em `triagem_resultados` e os scores do teste em `triagem_scores`. Um erro numa configuração é registrado e a execução continua. Com `retomar=True`, configurações já concluídas são puladas.
6. **Resumo:** média e desvio entre sementes, diferença para o baseline, sobreajuste e fração de safras com ganho.
7. **Bootstrap pareado** nas `n_candidatos_bootstrap` configurações com maior diferença para o baseline.
8. **Lista curta:** entre as técnicas elegíveis com diferença média positiva, o nível de maior média de cada técnica; ficam as `tamanho_lista_curta` maiores diferenças. Cada uma é gravada como `treino_<tecnica>`, junto com `treino_baseline`.

A triagem é um filtro, não a decisão: a decisão com intervalo de confiança é feita no bloco 3, no OOT de seleção.

O KS aparece em escala de 0 a 1. A coluna `metrica_decisao` é a métrica escolhida no bloco 1.

In [ ]:
ESQUEMA_FIXO_INICIO = [
    ("config_id", T.StringType()), ("tecnica", T.StringType()), ("familia", T.StringType()),
    ("nivel", T.DoubleType()), ("semente", T.LongType()), ("status", T.StringType()), ("motivo", T.StringType()),
    ("n_positivos", T.DoubleType()), ("n_negativos", T.DoubleType()), ("n_sinteticos", T.DoubleType()),
    ("removidos", T.DoubleType()), ("beta", T.DoubleType()),
    ("metrica_decisao", T.DoubleType()), ("metrica_treino", T.DoubleType()), ("gap_treino_teste", T.DoubleType()),
    ("ks", T.DoubleType()), ("auc_roc", T.DoubleType()), ("gini", T.DoubleType()), ("pr_auc", T.DoubleType()),
]
ESQUEMA_FIXO_FIM = [("por_safra", T.StringType()), ("info", T.StringType()),
                    ("segundos", T.DoubleType()), ("momento", T.StringType())]


def esquema_resultados(taxas):
    campos = ESQUEMA_FIXO_INICIO + [(c, T.DoubleType()) for c in colunas_operacionais(taxas)] + ESQUEMA_FIXO_FIM
    return T.StructType([T.StructField(n, t, True) for n, t in campos])


def _id_config(tecnica, nivel, semente):
    return f"{tecnica}|{'-' if nivel is None else f'{nivel:g}'}|{semente}"


def _gravar(df, tabela, estado, chave_estado):
    modo = "overwrite" if estado[chave_estado] else "append"
    escrita = df.write.format(FORMATO_TABELA).mode(modo)
    if modo == "overwrite":
        escrita = escrita.option("overwriteSchema", "true")
    escrita.saveAsTable(tabela)
    estado[chave_estado] = False


def _linha_resultado(esquema, valores):
    linha = []
    for campo in esquema.fields:
        v = valores.get(campo.name)
        if isinstance(campo.dataType, T.DoubleType):
            v = None if v is None or (isinstance(v, float) and not np.isfinite(v)) else float(v)
        elif isinstance(campo.dataType, T.LongType):
            v = None if v is None else int(v)
        elif v is not None:
            v = str(v)
        linha.append(v)
    return tuple(linha)


def avaliar_aplicabilidade(base, parametros):
    n_pos, n_neg = base["n_pos"], base["n_neg"]
    n_dist = len(base["features_dist"])
    excluidas, niveis_validos = [], {}
    for nome, meta in TECNICAS.items():
        if nome == "baseline":
            continue
        motivo = None
        if meta["distancia"] and n_dist == 0:
            motivo = "Nenhuma feature contínua disponível para calcular distâncias."
        elif meta["driver"] and not base["vizinhanca_ok"]:
            motivo = (f"{n_pos:,} positivos no treino, acima de limite_positivos_vizinhanca "
                      f"({parametros['limite_positivos_vizinhanca']:,}).")
        elif nome in TECNICAS_SMOTE and n_pos < parametros["k_vizinhos"] + 1:
            motivo = f"São necessários pelo menos {parametros['k_vizinhos'] + 1} positivos."
        elif nome in ("iht", "negativos_confiaveis") and n_pos < parametros["n_folds_cross_fitting"]:
            motivo = "Positivos insuficientes para o cross-fitting."
        if motivo:
            excluidas.append({"tecnica": nome, "nivel": None, "motivo": motivo})
            continue
        if meta["niveis"] is None:
            niveis_validos[nome] = [None]
            continue
        validos = []
        for nivel in parametros[meta["niveis"]]:
            if meta["niveis"] == "niveis_subamostragem" and _n_manter(base, nivel) >= n_neg:
                excluidas.append({"tecnica": nome, "nivel": nivel,
                                  "motivo": "O nível manteria todos os negativos: não há o que subamostrar."})
            elif meta["niveis"] == "niveis_negativos_confiaveis" and int(builtins.round(nivel * n_neg)) < 1:
                excluidas.append({"tecnica": nome, "nivel": nivel, "motivo": "O nível não remove nenhum negativo."})
            elif nome in ("smote_tomek", "smote_enn") and n_pos * nivel > parametros["limite_positivos_vizinhanca"]:
                excluidas.append({"tecnica": nome, "nivel": nivel,
                                  "motivo": "Positivos após o SMOTE acima de limite_positivos_vizinhanca."})
            elif meta["niveis"] == "niveis_sobreamostragem" and nivel <= 1:
                excluidas.append({"tecnica": nome, "nivel": nivel, "motivo": "Multiplicador deve ser maior que 1."})
            else:
                validos.append(nivel)
        if validos:
            niveis_validos[nome] = validos
    return niveis_validos, pd.DataFrame(excluidas, columns=["tecnica", "nivel", "motivo"])


def montar_grade(niveis_validos, parametros):
    semente = parametros["semente"]
    sementes = [semente + 7919 * (i + 1) for i in range(parametros["sementes_grade"])]
    grade = []
    for nome, meta in TECNICAS.items():
        if nome == "baseline" or nome not in niveis_validos:
            continue
        for nivel in niveis_validos[nome]:
            for s in (sementes if meta["sorteio"] else [semente]):
                grade.append({"id": _id_config(nome, nivel, s), "tecnica": nome, "nivel": nivel, "semente": s})
    return grade


def _scores_avaliacao(base, proxy):
    """Scores do teste e da amostra de avaliação do treino, alinhados a teste_ids e amostra_ids."""
    p = base["parametros"]
    cache = base["cache_pandas"]
    if proxy["motor"] == "pandas":
        if "teste" not in cache:
            cache["teste"] = matriz_pandas(base["teste"], base["features"], manter=["_id_linha"])
            cache["amostra"] = matriz_pandas(base["amostra_treino"], base["features"], manter=["_id_linha"])
        return prever_pandas(proxy, cache["teste"]), prever_pandas(proxy, cache["amostra"])
    st = pontuar(proxy, base["teste"], ["_id_linha"], p).toPandas()
    sa = pontuar(proxy, base["amostra_treino"], ["_id_linha"], p).toPandas()
    return (pd.Series(st["score"].to_numpy(), index=st["_id_linha"].to_numpy()).reindex(base["teste_ids"]).to_numpy(),
            pd.Series(sa["score"].to_numpy(), index=sa["_id_linha"].to_numpy()).reindex(base["amostra_ids"]).to_numpy())


def _executar_configuracao(base, cfg, metrica, esquema, tabelas, estado):
    p = base["parametros"]
    taxas = p["taxas_alerta_relatorio"]
    meta = TECNICAS[cfg["tecnica"]]
    valores = {"config_id": cfg["id"], "tecnica": cfg["tecnica"], "familia": meta["familia"],
               "nivel": cfg["nivel"], "semente": cfg["semente"],
               "momento": datetime.now().isoformat(timespec="seconds")}
    inicio = time.time()
    resultado, importancias = None, {}
    try:
        resultado = aplicar_tecnica(base, cfg["tecnica"], cfg["nivel"], cfg["semente"], metrica)
        if "inaplicavel" in resultado:
            valores.update(status="inaplicavel", motivo=resultado["inaplicavel"])
        else:
            df_tecnica = resultado["df"].persist()
            proxy = treinar_proxy(df_tecnica, base["teste"], base["features"], base["alvo"], metrica,
                                  p["semente"], p, col_peso=resultado["col_peso"], cache=base["cache_pandas"])
            if cfg["tecnica"] == "baseline":
                importancias = importancias_proxy(proxy)
            s_teste, s_amostra = _scores_avaliacao(base, proxy)
            df_tecnica.unpersist()

            nome = NOME_METRICA[metrica]
            m_teste = calcular_metricas(base["y_teste"], s_teste, None, taxas)
            m_treino = calcular_metricas(base["y_amostra"], s_amostra, base["w_amostra"])
            valores.update(m_teste)
            valores.update(
                status="ok", n_positivos=resultado["n_pos"], n_negativos=resultado["n_neg"],
                n_sinteticos=resultado["n_sinteticos"], removidos=resultado.get("removidos"), beta=resultado["beta"],
                metrica_decisao=m_teste[nome], metrica_treino=m_treino[nome],
                gap_treino_teste=m_treino[nome] - m_teste[nome],
                por_safra=json.dumps(metricas_por_safra(base["y_teste"], s_teste, base["safra_teste"], metrica)),
                info=json.dumps({"importancias": importancias, "motor": proxy["motor"],
                                 "iteracoes": proxy.get("iteracoes"), "memoria_mb": proxy.get("memoria_mb"),
                                 "parametros_ignorados": proxy.get("ignorados", [])}),
            )
            scores = spark.createDataFrame(pd.DataFrame({
                "config_id": cfg["id"], "_id_linha": base["teste_ids"].astype(np.int64), "score": s_teste}))
            _gravar(scores, tabelas["scores"], estado, "scores")
    except Exception as erro:
        valores.update(status="erro", motivo=f"{type(erro).__name__}: {erro}"[:2000])
    finally:
        if resultado is not None:
            liberar_resultado(resultado)
    valores["segundos"] = time.time() - inicio
    _gravar(spark.createDataFrame([_linha_resultado(esquema, valores)], esquema), tabelas["resultados"], estado, "resultados")
    return valores, importancias


def _resumir(resultados, metrica, base):
    ok = resultados[resultados["status"] == "ok"].copy()
    ok["nivel_txt"] = ok["nivel"].map(lambda v: "-" if pd.isna(v) else f"{v:g}")
    linha_base = ok[ok["tecnica"] == "baseline"]
    if linha_base.empty:
        raise RuntimeError("O baseline não foi concluído; veja a tabela de resultados.")
    valor_base = float(linha_base["metrica_decisao"].mean())
    safras_base = json.loads(linha_base["por_safra"].iloc[0])

    linhas = []
    for (tecnica, nivel_txt), g in ok.groupby(["tecnica", "nivel_txt"], sort=False):
        safras = pd.DataFrame([json.loads(s) for s in g["por_safra"]]).mean()
        comparaveis = [s for s in safras.index if s in safras_base and np.isfinite(safras[s]) and np.isfinite(safras_base[s])]
        ganho = [safras[s] > safras_base[s] for s in comparaveis]
        linha = {
            "tecnica": tecnica, "familia": TECNICAS[tecnica]["familia"], "nivel": g["nivel"].iloc[0],
            "execucoes": len(g), "metrica_media": g["metrica_decisao"].mean(),
            "metrica_desvio": g["metrica_decisao"].std(ddof=1) if len(g) > 1 else np.nan,
            "diferenca_baseline": g["metrica_decisao"].mean() - valor_base,
            "fracao_safras_com_ganho": float(np.mean(ganho)) if ganho else np.nan,
            "ks": g["ks"].mean(), "auc_roc": g["auc_roc"].mean(), "pr_auc": g["pr_auc"].mean(),
            "gap_treino_teste": g["gap_treino_teste"].mean(),
            "n_positivos": g["n_positivos"].mean(), "n_negativos": g["n_negativos"].mean(),
            "beta": g["beta"].mean(), "elegivel_lista_curta": TECNICAS[tecnica]["elegivel"],
            "config_ids": list(g["config_id"]),
        }
        for c in colunas_operacionais(base["parametros"]["taxas_alerta_relatorio"]):
            linha[c] = g[c].mean()
        linhas.append(linha)
    resumo = pd.DataFrame(linhas).sort_values("diferenca_baseline", ascending=False).reset_index(drop=True)
    return resumo, valor_base


def _bootstrap_candidatos(resumo, base, metrica, tabelas):
    p = base["parametros"]
    candidatos = resumo[resumo["tecnica"] != "baseline"].head(p["n_candidatos_bootstrap"])
    if candidatos.empty:
        return pd.DataFrame()
    ids_base = resumo.loc[resumo["tecnica"] == "baseline", "config_ids"].iloc[0]
    todos = list(ids_base) + [i for ids in candidatos["config_ids"] for i in ids]
    lista_ids = spark.createDataFrame(pd.DataFrame({"config_id": todos}))
    scores = (spark.table(tabelas["scores"]).join(lista_ids, on="config_id", how="left_semi").toPandas()
              .drop_duplicates(["config_id", "_id_linha"], keep="last"))
    por_config = {cid: pd.Series(g["score"].to_numpy(), index=g["_id_linha"].to_numpy()).reindex(base["teste_ids"]).to_numpy()
                  for cid, g in scores.groupby("config_id")}
    s_base = [por_config[i] for i in ids_base]
    linhas = []
    for _, c in candidatos.iterrows():
        r = bootstrap_pareado(base["y_teste"], base["codigos_chave_teste"], [por_config[i] for i in c["config_ids"]],
                              s_base, metrica, p["n_bootstrap"], p["semente"])
        linhas.append({"tecnica": c["tecnica"], "nivel": c["nivel"], **r})
    return pd.DataFrame(linhas)


def _escolher_lista_curta(resumo, bootstrap, parametros):
    elegiveis = resumo[(resumo["tecnica"] != "baseline") & resumo["elegivel_lista_curta"] & (resumo["diferenca_baseline"] > 0)]
    melhores = (elegiveis.sort_values("metrica_media", ascending=False)
                .drop_duplicates("tecnica").sort_values("diferenca_baseline", ascending=False)
                .head(parametros["tamanho_lista_curta"]))
    return melhores


def executar_triagem(contexto, **sobrescritos):
    if not contexto.get("modulo_ativo"):
        log_pipeline("Módulo de desbalanceamento inativo: a triagem não foi executada.")
        return contexto
    parametros = montar_parametros_bloco2(contexto, **sobrescritos)
    metrica = contexto["metrica"]
    validar_metrica(metrica, True)
    if metrica not in NOME_METRICA:
        raise ValueError(f"A triagem só usa {sorted(NOME_METRICA)}.")
    registro = contexto["registro"]
    taxas = parametros["taxas_alerta_relatorio"]
    esquema = esquema_resultados(taxas)

    # 1. Bases
    base = preparar_tabelas_triagem(contexto, parametros, metrica)
    tabelas = {"resultados": nome_tabela(base["destino"], base["prefixo"], "triagem_resultados"),
               "scores": nome_tabela(base["destino"], base["prefixo"], "triagem_scores")}
    feitos = {}
    if base["reaproveitar"]:
        try:
            anteriores = spark.table(tabelas["resultados"])
            if [(c.name, c.dataType) for c in anteriores.schema.fields] == [(c.name, c.dataType) for c in esquema.fields]:
                feitos = {r["config_id"]: r for r in anteriores.collect() if r["status"] in ("ok", "inaplicavel")}
        except Exception:
            feitos = {}
    estado = {"resultados": not feitos, "scores": not feitos}
    for t in tabelas.values():
        registro["tabelas"].append(t)

    # 2. Baseline
    id_base = _id_config("baseline", None, parametros["semente"])
    if id_base in feitos:
        importancias = json.loads(feitos[id_base]["info"]).get("importancias", {})
        log_pipeline("Baseline já concluído na execução anterior.")
    else:
        log_pipeline("Treinando o baseline.")
        valores, importancias = _executar_configuracao(
            base, {"id": id_base, "tecnica": "baseline", "nivel": None, "semente": parametros["semente"]},
            metrica, esquema, tabelas, estado)
        if valores["status"] != "ok":
            raise RuntimeError(f"O baseline falhou: {valores.get('motivo')}")
        log_pipeline(f"Baseline: {metrica} = {valores['metrica_decisao']:.4f} no teste "
                     f"({valores['segundos']:.0f} s por treino).")

    # 3. Vizinhança e aplicabilidade
    features_dist = escolher_features_distancia(base["features"], base["categoricas"], importancias,
                                                parametros["max_features_distancia"])
    preparar_vizinhanca(base, features_dist)
    niveis_validos, excluidas = avaliar_aplicabilidade(base, parametros)

    # 4 e 5. Grade e execução
    grade = montar_grade(niveis_validos, parametros)
    pendentes = [c for c in grade if c["id"] not in feitos]
    log_pipeline(f"Grade: {len(grade) + 1} treinos (com o baseline); {len(pendentes)} pendentes.")
    inicio = time.time()
    for n, cfg in enumerate(pendentes, start=1):
        valores, _ = _executar_configuracao(base, cfg, metrica, esquema, tabelas, estado)
        decorrido = time.time() - inicio
        restante = decorrido / n * (len(pendentes) - n)
        detalhe = (f"{metrica} = {valores['metrica_decisao']:.4f}" if valores["status"] == "ok"
                   else f"{valores['status']}: {valores.get('motivo', '')[:200]}")
        log_pipeline(f"[{n}/{len(pendentes)}] {cfg['id']}: {detalhe} | restante estimado: {restante / 60:.0f} min")

    # 6. Resumo
    resultados = spark.table(tabelas["resultados"]).toPandas()
    resumo, valor_base = _resumir(resultados, metrica, base)

    # 7. Bootstrap
    log_pipeline("Bootstrap pareado dos candidatos.")
    bootstrap = _bootstrap_candidatos(resumo, base, metrica, tabelas)

    # 8. Lista curta
    escolhidas = _escolher_lista_curta(resumo, bootstrap, parametros)
    lista_curta = []
    colunas_finais = [coluna(c) for c in _colunas_saida(base)]
    t_base = salvar_tabela(base["treino"].select(*colunas_finais), base["destino"], base["prefixo"], "treino_baseline", registro)
    lista_curta.append({"tecnica": "baseline", "nivel": None, "semente": parametros["semente"], "beta": 1.0,
                        "diferenca_baseline": 0.0, "tabela": nome_tabela(base["destino"], base["prefixo"], "treino_baseline")})
    for _, e in escolhidas.iterrows():
        tecnica = e["tecnica"]
        nivel = None if pd.isna(e["nivel"]) else float(e["nivel"])
        semente = parametros["semente"] + 7919 if TECNICAS[tecnica]["sorteio"] else parametros["semente"]
        resultado = aplicar_tecnica(base, tecnica, nivel, semente, metrica)
        nome = f"treino_{tecnica}"
        salvar_tabela(resultado["df"], base["destino"], base["prefixo"], nome, registro)
        liberar_resultado(resultado)
        linha_boot = bootstrap[(bootstrap["tecnica"] == tecnica) & (bootstrap["nivel"].fillna(-1) == (nivel if nivel is not None else -1))] if len(bootstrap) else pd.DataFrame()
        lista_curta.append({
            "tecnica": tecnica, "nivel": nivel, "semente": semente, "beta": resultado["beta"],
            "diferenca_baseline": float(e["diferenca_baseline"]),
            "ic_inferior": float(linha_boot["ic_inferior"].iloc[0]) if len(linha_boot) else None,
            "ic_superior": float(linha_boot["ic_superior"].iloc[0]) if len(linha_boot) else None,
            "tabela": nome_tabela(base["destino"], base["prefixo"], nome),
        })
    if len(lista_curta) == 1:
        log_pipeline("Nenhuma técnica elegível superou o baseline no teste: a lista curta contém só o baseline.")
    else:
        log_pipeline("Lista curta: " + ", ".join(
            f"{x['tecnica']}" + ("" if x["nivel"] is None else f" ({x['nivel']:g})") for x in lista_curta[1:]))

    registro["triagem"] = {
        "metrica": metrica, "valor_baseline_teste": valor_base, "features_distancia": features_dist,
        "n_configuracoes": len(grade) + 1, "tabelas": tabelas,
        "lista_curta": [{k: v for k, v in x.items()} for x in lista_curta],
    }
    contexto.update({
        "parametros_bloco2": parametros,
        "features_distancia": features_dist,
        "resultados_triagem": resultados,
        "resumo_triagem": resumo.drop(columns=["config_ids"]),
        "bootstrap_triagem": bootstrap,
        "lista_curta": lista_curta,
        "tecnicas_excluidas": excluidas,
        "tabelas_triagem": {"treino": nome_tabela(base["destino"], base["prefixo"], "base_treino"),
                            "teste": nome_tabela(base["destino"], base["prefixo"], "base_teste"), **tabelas},
    })
    return contexto

## Célula 9 — Exemplo de uso

Rodar depois do bloco 1, na mesma sessão. Para um primeiro teste mais rápido, reduza a grade, por exemplo `sementes_grade=1` ou `niveis_subamostragem=[10]`.

Se a execução cair no meio, basta rodar a mesma célula de novo: com `retomar=True`, as configurações concluídas são puladas. Para recomeçar do zero, use `retomar=False`.

In [ ]:
# contexto = executar_triagem(contexto)
#
# display(contexto["resumo_triagem"])        # uma linha por técnica e nível, ordenada pela diferença para o baseline
# display(contexto["bootstrap_triagem"])     # intervalo de 95% da diferença para os principais candidatos
# display(contexto["tecnicas_excluidas"])    # técnicas e níveis que não se aplicam a esta base, com o motivo
# for item in contexto["lista_curta"]:
#     print(item)